In [1]:
import requests
import pandas as pd
import numpy as np
from openai import OpenAI

In [10]:
# =========================================================
# 1) CONFIGURAÇÃO DA LLM
# =========================================================

# Ajuste conforme seu ambiente
# Exemplo:
# - Cloudera AI Inference compatível com OpenAI
# - outro endpoint compatível
BASE_URL = "https://hol-cai-infe-services.hol-spag.z30z-14kp.cloudera.site/namespaces/serving-default/endpoints/llama-31-8b/v1"
API_KEY = "eyJraWQiOiIzYzhlNzA3OTEyZmI0NTA1ODE3NzE3YzMyOTU4MmQwMTFjYjlmNTAwIiwidHlwIjoiSldUIiwiYWxnIjoiUlMyNTYifQ.eyJzdWIiOiJ2Y29uZGUiLCJhdWQiOiJodHRwczovL2RlLnozMHotMTRrcC5jbG91ZGVyYS5zaXRlIiwiaXNzIjoiaHR0cHM6Ly9jb25zb2xlYXV0aC5jZHAuY2xvdWRlcmEuY29tL2Y4ZTE2ZDJkLTU3ZTYtNDNiYy04ZTUwLTc1YTMyZjc3ODJhZCIsImdyb3VwcyI6ImNkcF9maWVsZF9tYXJrZXRpbmdfYW1lciBob2wtcGljcGF5LWF3LWNkcC1hZG1pbi1ncm91cCBob2wtc3BhZ3Vhcy1pbnN0cnVjdG9yIF9jX21sX2FkbWluc181ZDgzNzY1ZiBfY19uaWZpX3JlZ2lzdHJ5X2FkbWluc18xMzFlN2JhMiBfY19oYmFzZV9hZG1pbnNfMTMxZTdiYTIgX2NfbWxfYWRtaW5zXzEzMWU3YmEyIF9jX2tub3hfYWRtaW5zXzEzMWU3YmEyIF9jX2Vudl9hc3NpZ25lZXNfMTMxZTdiYTIgX2NfbmlmaV9hZG1pbnNfMTMxZTdiYTIgX2NfZWZtX2FkbWluc18xMzFlN2JhMiBfY19tbF91c2Vyc18xMzFlN2JhMiBfY196ZXBwZWxpbl9hZG1pbnNfMTMxZTdiYTIgX2NfZGZfcHJvamVjdF9tZW1iZXJfNWFlMDVmN2UgX2NfcmFuZ2VyX2FkbWluc18xMzFlN2JhMiBfY19jbV9hZG1pbnNfMTMxZTdiYTIiLCJleHAiOjE3NzQyODUzNDAsInR5cGUiOiJ1c2VyIiwiZ2l2ZW5fbmFtZSI6IlZpdG9yIiwiaWF0IjoxNzc0MjgxNzQwLCJmYW1pbHlfbmFtZSI6IkNvbmRlIiwiZW1haWwiOiJ2Y29uZGVAY2xvdWRlcmEuY29tIn0.n-AWXe47dcYnaetxHKxskiBQnw9o2CjXdvrJnngsDpYzZkCs5w7zk40yYxMC5Mqiqy6IavI4znTYg-odnvlVddQ_1HwOanTkn_bFrNZWjdbVqOYxHgUlTdtwqRjyQVfvRTGejwdWwX2UtJCAC1lIxGIcUcmFYR5ZGS88flvgzL4T3kyDpsJvi-rjE4goQ_f8EPoRLJwJbAv04vxItMLDL-6wa7XxBpu4e3BDnqDSwCrnrL1JFN4h2Egp5IIkYaDX3PL54OPHU7MHXVZ42J3V5-KaCl4wyYMm12v0pJIqBlIE7kQPHKqAdhBfw1Ld2l9kpyC5ukxQVSpXh04KhAZN1Q"
MODEL_NAME = "meta/llama-3.1-8b-instruct"

client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

In [11]:
# =========================================================
# 2) COLETA DOS DADOS
# =========================================================

def get_rain_data(hours=6):
    """
    Baixa dados da API de chuva.
    """
    url = "https://apps.spaguas.sp.gov.br/sibh/api/v2/measurements/now"
    params = {
        "station_type_id": 2,
        "hours": hours,
        "show_all": "true",
        "serializer": "complete",
        "public": "true"
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.json().get("measurements", [])


def get_level_data():
    """
    Baixa dados da API de nível.
    """
    url = "https://apps.spaguas.sp.gov.br/sibh/api/v2/measurements/now_flu"

    params = [
        ("references[]", "extravasation"),
        ("references[]", "emergency"),
        ("references[]", "alert"),
        ("references[]", "attention"),
        ("with_one_ref", "true")
    ]

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()

    if isinstance(data, dict):
        for key in ["measurements", "data", "results"]:
            if key in data and isinstance(data[key], list):
                return data[key]

    if isinstance(data, list):
        return data

    return []



In [12]:
# =========================================================
# 3) PREPARAÇÃO DOS DATAFRAMES
# =========================================================

def prepare_rain_df(rain_data):
    """
    Normaliza os dados de chuva.
    """
    df = pd.json_normalize(rain_data)

    for col in ["station_name", "city", "value"]:
        if col not in df.columns:
            df[col] = None

    df["rain_value"] = pd.to_numeric(df["value"], errors="coerce")

    keep_cols = [c for c in [
        "station_name",
        "city",
        "rain_value",
        "max_value",
        "max_date",
        "min_value",
        "min_date"
    ] if c in df.columns]

    return df[keep_cols].copy()


def prepare_level_df(level_data):
    """
    Normaliza os dados de nível.
    """
    df = pd.json_normalize(level_data)

    for col in ["station_name", "city", "value", "attention", "alert", "emergency", "extravasation"]:
        if col not in df.columns:
            df[col] = None

    df["level_value"] = pd.to_numeric(df["value"], errors="coerce")

    for ref_col in ["attention", "alert", "emergency", "extravasation"]:
        df[ref_col] = pd.to_numeric(df[ref_col], errors="coerce")

    keep_cols = [c for c in [
        "station_name",
        "city",
        "level_value",
        "attention",
        "alert",
        "emergency",
        "extravasation"
    ] if c in df.columns]

    return df[keep_cols].copy()



In [13]:
# =========================================================
# 4) MODELO SIMPLES DE RISCO
# =========================================================

def calculate_level_ratio(row):
    """
    Mede o quão próximo o nível atual está das referências.
    """
    level_value = row.get("level_value", np.nan)

    if pd.isna(level_value):
        return 0.0

    score = 0.0

    for ref in ["attention", "alert", "emergency", "extravasation"]:
        ref_value = row.get(ref, np.nan)

        if not pd.isna(ref_value) and ref_value != 0:
            score = max(score, level_value / ref_value)

    return min(score, 1.5)


def classify_risk(score):
    """
    Classifica o score final.
    """
    if score >= 0.95:
        return "CRÍTICO"
    elif score >= 0.75:
        return "ALTO"
    elif score >= 0.45:
        return "MÉDIO"
    else:
        return "BAIXO"


def build_risk_model(hours=6):
    """
    Monta a base final de risco combinando chuva + nível.
    """
    rain_df = prepare_rain_df(get_rain_data(hours))
    level_df = prepare_level_df(get_level_data())

    merged = pd.merge(
        rain_df,
        level_df,
        on=["station_name", "city"],
        how="outer"
    )

    merged["rain_value"] = merged["rain_value"].fillna(0)
    merged["level_value"] = merged["level_value"].fillna(0)

    max_rain = merged["rain_value"].max() if len(merged) > 0 else 0
    merged["rain_score"] = merged["rain_value"] / max_rain if max_rain > 0 else 0

    merged["level_score"] = merged.apply(calculate_level_ratio, axis=1)

    merged["risk_score"] = (
        merged["rain_score"] * 0.4 +
        merged["level_score"] * 0.6
    )

    merged["risk_class"] = merged["risk_score"].apply(classify_risk)

    merged = merged.sort_values("risk_score", ascending=False)

    return merged




In [14]:
# =========================================================
# 5) TRANSFORMAR DADOS EM CONTEXTO PARA A LLM
# =========================================================

def build_context_for_llm(df, top_n=30):
    """
    Converte os registros mais relevantes em texto.
    Esse texto será enviado junto com a pergunta.
    """
    df_top = df.head(top_n).copy()

    lines = []
    lines.append("DADOS DAS ESTAÇÕES:\n")

    for _, row in df_top.iterrows():
        station = row.get("station_name", "N/D")
        city = row.get("city", "N/D")
        rain = row.get("rain_value", 0)
        level = row.get("level_value", 0)
        risk_score = row.get("risk_score", 0)
        risk_class = row.get("risk_class", "N/D")
        attention = row.get("attention", None)
        alert = row.get("alert", None)
        emergency = row.get("emergency", None)
        extravasation = row.get("extravasation", None)

        line = (
            f"Estação: {station} | "
            f"Cidade: {city} | "
            f"Chuva: {rain:.2f} | "
            f"Nível: {level:.2f} | "
            f"Score de risco: {risk_score:.2f} | "
            f"Classe de risco: {risk_class} | "
            f"Atenção: {attention} | "
            f"Alerta: {alert} | "
            f"Emergência: {emergency} | "
            f"Extravasamento: {extravasation}"
        )

        lines.append(line)

    return "\n".join(lines)




In [15]:
# =========================================================
# 6) FUNÇÃO PARA PERGUNTAR À LLM
# =========================================================

def ask_llm_about_data(question, df, top_n=30):
    """
    Envia para a LLM:
    - os dados em formato texto
    - a pergunta do usuário

    A LLM deve responder apenas com base no contexto.
    """
    context = build_context_for_llm(df, top_n=top_n)

    prompt = f"""
Você é um analista hidrológico.

Responda apenas com base no contexto abaixo.
Não invente dados.
Se a resposta não estiver claramente no contexto, diga isso.
Se possível, cite nomes de estações e cidades.

Contexto:
{context}

Pergunta do usuário:
{question}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "Você responde de forma simples, objetiva e baseada somente nos dados fornecidos."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content




In [16]:
# =========================================================
# 7) LOOP INTERATIVO DE PERGUNTAS
# =========================================================

def run_chat():
    """
    Executa um mini chat no terminal.
    O usuário pode fazer várias perguntas sobre os dados.
    """
    print("Baixando dados e montando modelo de risco...")
    df_risk = build_risk_model(hours=6)

    print("\nBase pronta.")
    print("Você já pode fazer perguntas sobre os dados.")
    print("Digite 'sair' para encerrar.\n")

    while True:
        question = input("Pergunta: ").strip()

        if question.lower() in ["sair", "exit", "quit"]:
            print("Encerrando chat.")
            break

        try:
            answer = ask_llm_about_data(question, df_risk, top_n=30)

            print("\nResposta da LLM:")
            print(answer)
            print("\n" + "=" * 80 + "\n")

        except Exception as e:
            print(f"Erro ao consultar a LLM: {e}")



Você poderá perguntar coisas como:

- “Quais estações estão em maior risco?”
- “Existe alguma estação com chuva alta e nível baixo?”
- “Quais cidades parecem mais preocupantes?”
- “Explique a diferença entre as 3 estações mais críticas.”
- “Há indício de extravasamento?”

In [18]:
# =========================================================
# 8) EXECUÇÃO
# =========================================================

if __name__ == "__main__":
    run_chat()



Baixando dados e montando modelo de risco...

Base pronta.
Você já pode fazer perguntas sobre os dados.
Digite 'sair' para encerrar.



Pergunta:  Me de informações da Barra do Turvo.



Resposta da LLM:
Não há informações sobre a Barra do Turvo no contexto fornecido.




Pergunta:  Quais estações estão em maior risco?



Resposta da LLM:
As estações que estão em maior risco são aquelas com Score de risco igual a 0.90 e Classe de risco igual a ALTO.

As estações que atendem a esses critérios são:

- Estação: São José dos Campos (UHE Jaguari Fazenda Santana) | Cidade: São José dos Campos
- Estação: Rancharia (PCH Pari Barramento) | Cidade: Rancharia




Pergunta:  Me de informações de São José dos Campos.



Resposta da LLM:
A estação de São José dos Campos está localizada na UHE Jaguari Fazenda Santana. 

- Cidade: São José dos Campos
- Chuva: 0.00
- Nível: 1545.00
- Score de risco: 0.90
- Classe de risco: ALTO
- Atenção: 447.0
- Alerta: 458.0
- Emergência: nan
- Extravasamento: nan




KeyboardInterrupt: Interrupted by user